In [1]:
import sxobsplan
from pathlib import Path
import pandas as pd
import directory # type: ignore

In [2]:
EPH_DIR = directory.EPH_DIR
EPH_DIR.mkdir(exist_ok=True, parents=True)

# Small-Body Data Source: JPL SBDB Query
# https://ssd.jpl.nasa.gov/tools/sbdb_query.html
fpath_sbdb = directory.SBDB_DIR / "sbdb_query_results_comet_ver260530.csv"
df_sbdb = pd.read_csv(fpath_sbdb)

# Filter database before queries
mask  = (df_sbdb.prefix != "D") # Remove destroyed comets
mask &= ~((df_sbdb.prefix == "C") & (df_sbdb.epoch_cal <= "2021-01-01")) # Old comets

df_sbdb_filtered = df_sbdb[mask]
df_sbdb_filtered.info()
df_sbdb_filtered.head()

<class 'pandas.core.frame.DataFrame'>
Index: 988 entries, 0 to 3894
Data columns (total 79 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   spkid           988 non-null    int64  
 1   full_name       988 non-null    object 
 2   pdes            988 non-null    object 
 3   name            987 non-null    object 
 4   prefix          988 non-null    object 
 5   neo             95 non-null     object 
 6   pha             0 non-null      float64
 7   sats            988 non-null    int64  
 8   H               2 non-null      float64
 9   G               0 non-null      float64
 10  M1              980 non-null    float64
 11  M2              407 non-null    float64
 12  K1              980 non-null    float64
 13  K2              407 non-null    float64
 14  PC              404 non-null    float64
 15  diameter        96 non-null     float64
 16  extent          2 non-null      object 
 17  albedo          15 non-null     float64

,spkid,full_name,pdes,name,prefix,neo,pha,sats,H,G,...,rms,two_body,A1,A1_sigma,A2,A2_sigma,A3,A3_sigma,DT,DT_sigma
0,1000036,1P/Halley,1P,Halley,P,Y,NaN,0,NaN,NaN,...,0.61032,NaN,4.900000e-10,4.000000e-11,1.600000e-10,4.600000e-15,NaN,NaN,NaN,NaN
1,1000025,2P/Encke,2P,Encke,P,Y,NaN,0,NaN,NaN,...,0.44859,NaN,2.300000e-10,6.100000e-11,-6.500000e-13,4.000000e-12,NaN,NaN,NaN,NaN
3,1000026,4P/Faye,4P,Faye,P,NaN,NaN,0,NaN,NaN,...,0.54304,NaN,4.000000e-09,1.200000e-10,3.200000e-10,6.300000e-11,-7.100000e-10,4.200000e-11,-37.7,2.17
5,1000017,6P/d'Arrest,6P,d'Arrest,P,NaN,NaN,0,NaN,NaN,...,0.63861,NaN,1.800000e-10,2.700000e-10,-8.500000e-10,1.200000e-10,NaN,NaN,NaN,NaN
6,1000069,7P/Pons-Winnecke,7P,Pons-Winnecke,P,Y,NaN,0,NaN,NaN,...,0.66919,NaN,6.400000e-11,3.600000e-13,-5.200000e-12,5.000000e-13,-1.000000e-10,2.800000e-12,150.0,5.50


In [4]:
all_pdes_list = df_sbdb_filtered['pdes'].tolist()
epochs = {"start": "2026-01-01", "stop": "2028-12-31", "step": "2d"}

# 2. Identify missing targets (Skip existing ephemerides)
pdes_to_query = []
for pdes in all_pdes_list:
    safe_name = "".join(pdes.split())
    fpath_eph = EPH_DIR / f"{safe_name}.csv"
    
    if not fpath_eph.exists():
        pdes_to_query.append(pdes)

print(f"Total targets: {len(all_pdes_list)}")
print(f"Already downloaded: {len(all_pdes_list) - len(pdes_to_query)}")
print(f"To query: {len(pdes_to_query)}")

# 3. Fetch concurrently and save
if pdes_to_query:
    # Fetch all missing targets simultaneously
    stacked_eph = sxobsplan.batch_query(
        designations=pdes_to_query,
        epochs=epochs,
        quantities="1,3,9,18,19,20,22,23,24,25,27,28,29,33,43",
        location="500",
        max_workers=8,     # Optimal concurrency for JPL Horizons
        progress=True
    )
    
    if len(stacked_eph) > 0:
        # Group the stacked astropy table by the 'Name' column
        grouped_eph = stacked_eph.group_by('Name')
        
        # Iterate through each comet's individual sub-table and save it
        for key, group in zip(grouped_eph.groups.keys, grouped_eph.groups):
            pdes = key['Name']
            safe_name = "".join(pdes.split())
            fpath_eph = EPH_DIR / f"{safe_name}.csv"
            
            group.write(fpath_eph, format='ascii.csv', overwrite=False)
            
        print(f"\nSuccessfully saved {len(grouped_eph.groups)} new ephemerides.")
    else:
        print("\nFailed to fetch any new ephemerides.")
else:
    print("\nAll target ephemerides are up to date! Nothing to do.")

Total targets: 988
Already downloaded: 980
To query: 8


Batch Querying (Parallel): 100%|██████████| 8/8 [00:04<00:00,  1.74it/s]


Successfully saved 6 new ephemerides.
